# Electrodynamics from Scratch — Lecture 8: The Magnetic Moment and the Vector Potential

A steady current distribution $\mathbf{J}$ produces a magnetic field $\mathbf{B}$ that satisfies $\nabla \cdot \mathbf{B} = 0$ and $\nabla \times \mathbf{B} = \mu_0 \mathbf{J}$. Because the divergence of $\mathbf{B}$ vanishes everywhere, $\mathbf{B}$ can always be written locally as the curl of a **vector potential**,
$$\mathbf{B} = \nabla \times \mathbf{A}.$$
In the Coulomb gauge ($\nabla \cdot \mathbf{A} = 0$) the potential of a thin wire carrying a steady current $I$ is the magnetostatic analogue of Coulomb's law:
$$\mathbf{A}(\mathbf{r}) = \frac{\mu_0 I}{4\pi} \oint \frac{d\boldsymbol{\ell}'}{|\mathbf{r} - \mathbf{r}'|},$$
and the field is recovered by taking the curl, $\mathbf{B} = \nabla \times \mathbf{A}$ (this is Ampère's law in potential form; the Biot–Savart law follows directly).

Far from the loop, $1/|\mathbf{r}-\mathbf{r}'|$ can be expanded in powers of $R/r$, and the leading term is
$$\mathbf{A}(\mathbf{r}) \simeq \frac{\mu_0}{4\pi}\, \frac{\mathbf{m} \times \hat{\mathbf{r}}}{r^2},$$
where $\mathbf{m} = I\,\mathbf{a}$ is the **magnetic dipole moment** ($\mathbf{a}$ is the vector area of the loop). For a circular loop of radius $R$ in the $xy$-plane carrying a counterclockwise current, $\mathbf{m} = I\pi R^2\,\hat{\mathbf{z}}$. The corresponding field is that of a perfect point dipole,
$$\mathbf{B}_{\rm dip} = \frac{\mu_0}{4\pi r^3}\left[3(\mathbf{m}\cdot\hat{\mathbf{r}})\hat{\mathbf{r}} - \mathbf{m}\right],$$
which on the symmetry axis reduces to $B_z = \dfrac{\mu_0}{4\pi}\dfrac{2m}{z^3}$.

In this notebook we compute $\mathbf{m}$ for circular loops and a solenoid, evaluate the vector potential of a wire loop numerically, verify its $1/r^2$ decay and its matching to the point-dipole form at large distance, and compare field-line pictures of the exact loop field with the ideal dipole field.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import electrodynamics_from_scratch as eds

mu0 = eds.MU_0
print(f"mu0 = {mu0:.4e} N/A^2")

**The magnetic moment of a loop.** For a planar loop the moment is $\mathbf{m} = I\,\mathbf{a}$: magnitude equal to current times area, direction given by the right-hand rule. We build circular loops of several radii and currents with `eds.circular_loop_curve` and compute $\mathbf{m}$ with `eds.magnetic_dipole_moment`; every entry should reproduce $m = I\pi R^2$ along $\hat{\mathbf{z}}$.

In [ ]:
print(f"{'R':>5} {'I':>5} {'|m| numeric':>14} {'I*pi*R^2':>12} {'rel err':>10}")
for R, I in [(0.5, 2.0), (1.0, 1.0), (1.5, 3.0), (2.0, 0.5)]:
    curve = eds.circular_loop_curve(radius=R, center=np.zeros(3), n=200)
    m = eds.magnetic_dipole_moment(curve, I, closed=True)
    m_exact = I * np.pi * R**2
    rel = abs(np.linalg.norm(m) - m_exact) / m_exact
    print(f"{R:5.2f} {I:5.2f} {np.linalg.norm(m):14.6e} {m_exact:12.6e} {rel:10.2e}")

m1 = eds.magnetic_dipole_moment(eds.circular_loop_curve(radius=1.0, center=np.zeros(3)), 1.0, closed=True)
print("\nm for R=1, I=1 (counterclockwise in the xy-plane):", m1)
print("points along +z as expected:", np.allclose(m1, [0.0, 0.0, np.pi]))

**The vector potential of a loop.** `eds.vector_potential_wire` evaluates $\mathbf{A}(\mathbf{r}) = \frac{\mu_0 I}{4\pi}\oint d\boldsymbol{\ell}'/|\mathbf{r}-\mathbf{r}'|$ on a discretised wire. In the midplane ($z=0$) the potential is purely azimuthal, and far away it must fall off as $|A| \approx \mu_0 m/(4\pi\rho^2)$. We sample points along the $x$-axis, check the direction, and fit a power law to the decay.

In [ ]:
R, I = 1.0, 1.0
curve = eds.circular_loop_curve(radius=R, center=np.zeros(3), n=200)

rho = np.linspace(2.0, 10.0, 9)
pts = np.stack([rho, np.zeros_like(rho), np.zeros_like(rho)], axis=1)
A = eds.vector_potential_wire(curve, I, pts, closed=True)

print("A at (rho=2, 0, 0):", A[0])
print("purely azimuthal (along +y):",
      np.allclose(A[0], [0.0, A[0][1], 0.0], rtol=1e-8, atol=1e-20))

A_mag = np.linalg.norm(A, axis=1)
amp, exponent = eds.power_law_fit(rho, A_mag)
print(f"fitted |A| = {amp:.3e} * rho^{exponent:.3f}   (dipole prediction: exponent -2)")

A_dip_ref = mu0 * I * np.pi * R**2 / (4 * np.pi * rho**2)
fig, ax = eds.plot_decay(rho, {"numeric $|A_\\phi|$": A_mag,
                               "dipole reference $\\propto 1/\\rho^2$": A_dip_ref},
                         loglog=True, title="Vector potential of a current loop (midplane)")
ax.set_xlabel(r"$\rho$")
ax.set_ylabel(r"$|A_\phi|$ [T m]")
fig

**Matching the point-dipole form.** The far-field prediction is $\mathbf{A}_{\rm dip} = \frac{\mu_0}{4\pi}\frac{\mathbf{m}\times\hat{\mathbf{r}}}{r^2}$ with $\mathbf{m} = I\pi R^2\hat{\mathbf{z}}$. We compare the numerically computed $\mathbf{A}$ against this formula at increasing distance; the relative error is set by the next term in the multipole expansion and should shrink roughly like $R/\rho$.

In [ ]:
m = eds.magnetic_dipole_moment(curve, I, closed=True)
print("m =", m, "   I*pi*R^2 =", I * np.pi * R**2)
print(f"\n{'rho':>6} {'rel. error of A vs dipole form':>32}")
for rmax in [3.0, 5.0, 10.0, 20.0, 40.0]:
    pts = np.array([[rmax, 0.0, 0.0]])
    A_num = eds.vector_potential_wire(curve, I, pts, closed=True)[0]
    rhat = pts[0] / np.linalg.norm(pts[0])
    A_dip = mu0 / (4 * np.pi) * np.cross(m, rhat) / rmax**2
    rel = np.linalg.norm(A_num - A_dip) / np.linalg.norm(A_dip)
    print(f"{rmax:6.1f} {rel:32.3e}")

**The same comparison for $\mathbf{B}$ on the axis.** The exact on-axis field of a loop is $B_z = \frac{\mu_0 I R^2}{2\,(R^2+z^2)^{3/2}}$, while the dipole approximation gives $B_z = \frac{\mu_0 I R^2}{2 z^3}$. We compute both with `eds.biot_savart` and `eds.magnetic_dipole_field` and plot them together; the two curves merge as $z/R$ grows.

In [ ]:
z = np.linspace(2.0, 20.0, 60)
pts = np.stack([np.zeros_like(z), np.zeros_like(z), z], axis=1)
B_loop = eds.biot_savart(curve, I, pts, closed=True)
B_dip = eds.magnetic_dipole_field(m, pts)
Bz_loop, Bz_dip = B_loop[:, 2], B_dip[:, 2]

_, exp_axis = eds.power_law_fit(z, Bz_loop)
print(f"on-axis |B| decays like z^{exp_axis:.3f} (expected -3)")
rel_err = np.abs(Bz_loop - Bz_dip) / Bz_loop
print(f"relative dipole error: {rel_err[0]:.1%} at z = 2R, {rel_err[-1]:.1%} at z = 20R")

fig, ax = eds.plot_decay(z, {"exact loop $B_z$": Bz_loop, "point dipole $B_z$": Bz_dip},
                         loglog=True, title="Loop field on the axis vs point dipole")
ax.set_xlabel("$z/R$")
ax.set_ylabel("$B_z$ [T]")
fig

**Field lines: loop vs dipole.** Far from the source, the field lines of a current loop and of a point dipole are the same curves. We draw streamlines of $(B_x, B_z)$ in the $xz$-plane ($y = 0$) for the exact loop field (`eds.biot_savart`) and for the ideal dipole (`eds.magnetic_dipole_field`), wrapping each vector field in a meshgrid-based field function for `eds.plot_field_lines`.

In [ ]:
def loop_field_xz(x, y):
    """B of the loop in the plane y = 0; the plotting coordinate y stands for z."""
    X, Y = np.meshgrid(x, y)
    pts = np.stack([X.ravel(), np.zeros(X.size), Y.ravel()], axis=1)
    B = eds.biot_savart(curve, I, pts, closed=True)
    return B[:, 0].reshape(X.shape), B[:, 2].reshape(X.shape)

def dipole_field_xz(x, y):
    X, Y = np.meshgrid(x, y)
    pts = np.stack([X.ravel(), np.zeros(X.size), Y.ravel()], axis=1)
    B = eds.magnetic_dipole_field(m, pts)
    return B[:, 0].reshape(X.shape), B[:, 2].reshape(X.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
eds.plot_field_lines(loop_field_xz, extent=3.0, density=1.4, ax=ax1, title="current loop (exact)")
eds.plot_field_lines(dipole_field_xz, extent=3.0, density=1.4, ax=ax2, title="point dipole")
fig.tight_layout()
fig

**A solenoid as a stack of loops.** From far away a solenoid with $N$ turns is just $N$ coaxial loops in series, so its dipole moment should be $m = N I \pi R^2$ along $\hat{\mathbf{z}}$. We build one with `eds.solenoid_curve`, check the moment with `eds.magnetic_dipole_moment`, and draw the curve.

In [ ]:
R_s, L_s, N_turns, I_s = 0.5, 2.0, 20, 1.0
solenoid = eds.solenoid_curve(radius=R_s, length=L_s, turns=N_turns, n_per_turn=20)
m_sol = eds.magnetic_dipole_moment(solenoid, I_s, closed=True)
print("solenoid m          =", m_sol)
print("expected N*I*pi*R^2 =", N_turns * I_s * np.pi * R_s**2)

fig, ax = eds.plot_trajectory_3d(solenoid, title="Solenoid curve (20 turns)")
fig

## Exercises

1. **Gauge freedom.** Starting from $\nabla \cdot \mathbf{B} = 0$, argue that a vector potential with $\mathbf{B} = \nabla \times \mathbf{A}$ must exist (at least locally), and explain why the transformation $\mathbf{A} \to \mathbf{A} + \nabla\chi$ leaves $\mathbf{B}$ unchanged. You can see this numerically with `eds.gauge_transform` and `eds.check_gauge_invariance` on any grid pair.

2. **Square loop.** Build the curve of a square loop of side $s$ in the $xy$-plane as a `[M, 3]` array (the four corners, with the first point repeated at the end), compute its moment with `eds.magnetic_dipole_moment(..., closed=True)`, and compare with the exact result $m = I s^2$.

3. **Radius scaling of the potential.** For loops with $R = 0.5$ and $R = 2.0$ at the same current, sample the midplane vector potential as in the notebook and use `eds.power_law_fit` on each: verify that the fitted exponent stays near $-2$ while the fitted amplitude scales like $R^2$.

4. **Direction of $\mathbf{A}$.** Reverse the current ($I = -1$) and verify that $\mathbf{A}$ at $(\rho, 0, 0)$ flips sign but still points along $\hat{\mathbf{y}}$. Explain the original $+\hat{\mathbf{y}}$ direction with the right-hand rule applied to $d\boldsymbol{\ell}'$.

5. **How far is "far"?** Using `eds.biot_savart` and `eds.magnetic_dipole_field` on the axis, find the distance $z/R$ at which the dipole formula for $B_z$ is accurate to 1%. Compare with the prediction you get from expanding $(R^2 + z^2)^{-3/2}$ for $z \gg R$.

6. **Two loops.** Superpose the `eds.biot_savart` fields of two coaxial loops of radius $R$ separated by $R$ carrying parallel currents, draw the field lines of the combined field in the $xz$-plane with `eds.plot_field_lines`, and compare with the field of a single dipole of moment $2m$.